# Система рекомендаций книг: От базовых моделей до гибридной нейросетевой архитектуры

## Курсовой проект по рекомендательным системам

---

### Оглавление
1. **Загрузка и описание данных**
2. **Установление взаимосвязей между таблицами**
3. **Анализ дубликатов оценок**
4. **Базовые модели рекомендаций**
   - Модель популярности
   - Контентная модель
   - Item-Based коллаборативная фильтрация
   - SVD (матричная факторизация)
   - ALS (Alternating Least Squares)
5. **Расширенные признаки**
6. **Гибридная система рекомендаций**
7. **Оценка и сравнительный анализ**
8. **Продвинутая часть: нейросетевая архитектура**
9. **Сквозной пайплайн**

---

## Установка необходимых библиотек

In [ ]:
# Установка необходимых библиотек (раскомментируйте при необходимости)
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy
# !pip install implicit  # для ALS
# !pip install lightfm   # альтернатива для гибридных моделей
# !pip install torch     # для нейросетевого подхода

In [ ]:
# Импорт основных библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Библиотеки для ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix, lil_matrix
from scipy.sparse.linalg import svds

# Настройка отображения
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Библиотеки успешно загружены!")

---
# 1. Загрузка и описание данных

В данном проекте используются данные из платформы Goodreads. Датасет состоит из четырёх файлов:

| Файл | Описание |
|------|----------|
| `books.csv` | Информация о книгах (название, автор, рейтинг и т.д.) |
| `ratings.csv` | Оценки пользователей |
| `tags.csv` | Словарь тегов |
| `book_tags.csv` | Связь книг с тегами |

In [ ]:
# Загрузка данных
# Укажите путь к данным (для Google Colab загрузите файлы или используйте свой путь)
data_path = '/home/z/my-project/upload/'  # Измените на свой путь при необходимости

# Загружаем все таблицы
books = pd.read_csv(data_path + 'books.csv')
ratings = pd.read_csv(data_path + 'ratings.csv')
tags = pd.read_csv(data_path + 'tags.csv')
book_tags = pd.read_csv(data_path + 'book_tags.csv')

print("Данные успешно загружены!\n")

In [ ]:
# Функция для описания датасета
def describe_dataset(df, name):
    """Выводит подробную информацию о датасете"""
    print(f"\n{'='*60}")
    print(f"ДАТАСЕТ: {name}")
    print(f"{'='*60}")
    print(f"Размерность: {df.shape[0]} строк × {df.shape[1]} столбцов")
    print(f"\nСтолбцы: {list(df.columns)}")
    print(f"\nТипы данных:")
    print(df.dtypes)
    print(f"\nПропущенные значения:")
    print(df.isnull().sum())
    print(f"\nПервые 5 строк:")
    display(df.head())
    print(f"\nСтатистика:")
    display(df.describe())

# Описываем каждый датасет
describe_dataset(books, 'books.csv')
describe_dataset(ratings, 'ratings.csv')
describe_dataset(tags, 'tags.csv')
describe_dataset(book_tags, 'book_tags.csv')

### Выводы по разделу 1: Описание данных

**Основные характеристики датасетов:**

1. **books.csv** - содержит метаданные о книгах:
   - Уникальные идентификаторы: `id`, `book_id`, `best_book_id`, `work_id`
   - Информация о книге: `title`, `authors`, `original_publication_year`, `language_code`
   - Рейтинговые метрики: `average_rating`, `ratings_count`, `work_ratings_count`
   - Детализация оценок: `ratings_1` - `ratings_5` (количество оценок каждого балла)

2. **ratings.csv** - взаимодействия пользователей с книгами:
   - `book_id` - идентификатор книги
   - `user_id` - идентификатор пользователя
   - `rating` - оценка от 1 до 5

3. **tags.csv** - словарь тегов:
   - `tag_id` - уникальный идентификатор тега
   - `tag_name` - название тега (жанры, категории, пользовательские метки)

4. **book_tags.csv** - связь книг и тегов:
   - `goodreads_book_id` - идентификатор книги в Goodreads
   - `tag_id` - идентификатор тега
   - `count` - количество пользователей, применивших тег

---
# 2. Установление взаимосвязей между столбцами таблиц

Для корректной работы рекомендательной системы необходимо чётко определить связи между таблицами.

In [ ]:
# Анализ связей между таблицами
print("АНАЛИЗ СВЯЗЕЙ МЕЖДУ ТАБЛИЦАМИ\n")
print("="*60)

# 1. Связь между books и ratings
print("\n1. Связь books ↔ ratings:")
books_in_ratings = set(ratings['book_id'].unique())
books_in_books = set(books['book_id'].unique())
print(f"   - Уникальных book_id в ratings: {len(books_in_ratings)}")
print(f"   - Уникальных book_id в books: {len(books_in_books)}")
print(f"   - Пересечение: {len(books_in_ratings.intersection(books_in_books))}")

# Проверяем связь через goodreads_book_id
print(f"\n   Внимание: book_id в ratings может соответствовать goodreads_book_id в books!")

# Сопоставление book_id в ratings с goodreads_book_id в book_tags
books_in_book_tags = set(book_tags['goodreads_book_id'].unique())
print(f"\n   - goodreads_book_id в book_tags: {len(books_in_book_tags)}")
print(f"   - book_id в ratings: {len(books_in_ratings)}")
print(f"   - Пересечение: {len(books_in_ratings.intersection(books_in_book_tags))}")

# 2. Связь между tags и book_tags
print("\n2. Связь tags ↔ book_tags:")
tags_in_tags = set(tags['tag_id'].unique())
tags_in_book_tags = set(book_tags['tag_id'].unique())
print(f"   - Уникальных tag_id в tags: {len(tags_in_tags)}")
print(f"   - Уникальных tag_id в book_tags: {len(tags_in_book_tags)}")
print(f"   - Пересечение: {len(tags_in_tags.intersection(tags_in_book_tags))}")

# 3. Связь между books и book_tags
print("\n3. Связь books ↔ book_tags:")
print(f"   - goodreads_book_id в book_tags соответствует book_id в books")
print(f"   - Или может потребоваться маппинг через id/books_count")

In [ ]:
# Создание схемы связей
print("\n" + "="*60)
print("СХЕМА СВЯЗЕЙ МЕЖДУ ТАБЛИЦАМИ")
print("="*60)

# Определяем правильный маппинг
# Проверяем связь book_id в ratings с id в books
rating_book_ids = ratings['book_id'].unique()
book_ids = books['id'].unique()

print(f"\nbook_id в ratings: min={rating_book_ids.min()}, max={rating_book_ids.max()}")
print(f"id в books: min={book_ids.min()}, max={book_ids.max()}")

# Создаём маппинг для унификации идентификаторов
if len(rating_book_ids) == len(book_ids) and set(rating_book_ids) == set(book_ids):
    print("\n✓ book_id в ratings соответствует id в books")
    book_id_mapping = dict(zip(books['id'], books['book_id']))
else:
    print("\n⚠ Требуется создание маппинга для сопоставления")

print("""
\nСХЕМА ДАННЫХ:
┌─────────────┐         ┌─────────────┐
│   books     │         │   ratings   │
├─────────────┤         ├─────────────┤
│ id (PK)     │◄────────│ book_id (FK)│
│ book_id     │         │ user_id     │
│ title       │         │ rating      │
│ authors     │         └─────────────┘
│ ...         │
└──────┬──────┘
       │
       │ goodreads_book_id
       ▼
┌─────────────┐         ┌─────────────┐
│ book_tags   │────────►│    tags     │
├─────────────┤         ├─────────────┤
│ goodreads_  │         │ tag_id (PK) │
│ book_id(FK) │         │ tag_name    │
│ tag_id (FK) │         └─────────────┘
│ count       │
└─────────────┘
""")

In [ ]:
# Создание объединённого датасета с правильными связями
print("СОЗДАНИЕ ОБЪЕДИНЁННОГО ДАТАСЕТА\n")

# Создаём маппинг между id и book_id в таблице books
id_to_book_id = dict(zip(books['id'], books['book_id']))
book_id_to_id = dict(zip(books['book_id'], books['id']))

# Добавляем столбец для сопоставления с ratings
books['ratings_book_id'] = books['id']  # book_id в ratings соответствует id в books

# Проверяем, что маппинг работает
sample_books = books[['id', 'book_id', 'ratings_book_id', 'title']].head()
print("Пример маппинга идентификаторов:")
display(sample_books)

# Добавляем информацию о книгах к рейтингам
ratings_with_books = ratings.merge(
    books[['id', 'title', 'authors', 'average_rating', 'ratings_count']],
    left_on='book_id',
    right_on='id',
    how='left'
)

print(f"\nРазмер объединённого датасета: {ratings_with_books.shape}")
print("\nПример данных после объединения:")
display(ratings_with_books.head())

### Выводы по разделу 2: Взаимосвязи между таблицами

**Установленные связи:**

1. **books.id = ratings.book_id** - основной идентификатор книги для связи оценок с книгами
2. **book_tags.goodreads_book_id** - соответствует book_id в таблице books (внешний идентификатор Goodreads)
3. **book_tags.tag_id = tags.tag_id** - связь тегов с их названиями

**Созданные маппинги:**
- `id_to_book_id` - преобразование внутреннего id в goodreads book_id
- `book_id_to_id` - обратное преобразование

Эти связи будут использоваться для построения рекомендательных моделей.

---
# 3. Анализ дубликатов оценок

Важный этап предобработки - проверка на наличие дублирующихся оценок от одного пользователя одной книге.

In [ ]:
# Анализ дубликатов в оценках
print("АНАЛИЗ ДУБЛИКАТОВ ОЦЕНОК\n")
print("="*60)

# Общее количество записей
total_ratings = len(ratings)
print(f"Общее количество оценок: {total_ratings:,}")

# Количество уникальных пар (user_id, book_id)
unique_pairs = ratings.groupby(['user_id', 'book_id']).size().reset_index(name='count')
print(f"Уникальных пар (user_id, book_id): {len(unique_pairs):,}")

# Находим дубликаты
duplicates = unique_pairs[unique_pairs['count'] > 1]
print(f"\nПар с дубликатами: {len(duplicates):,}")

if len(duplicates) > 0:
    print(f"\nПримеры дубликатов:")
    display(duplicates.head(10))
    
    # Статистика дубликатов
    print(f"\nРаспределение количества дубликатов:")
    print(duplicates['count'].value_counts().sort_index())
    
    # Суммарное количество дублирующихся записей
    duplicate_count = duplicates['count'].sum() - len(duplicates)
    print(f"\nИзбыточных записей: {duplicate_count:,}")
    print(f"Процент дубликатов: {duplicate_count/total_ratings*100:.4f}%")

In [ ]:
# Детальный анализ дубликатов
if len(duplicates) > 0:
    print("\nДЕТАЛЬНЫЙ АНАЛИЗ ДУБЛИКАТОВ\n")
    print("="*60)
    
    # Проверяем, различаются ли оценки у дубликатов
    for idx, row in duplicates.head(5).iterrows():
        user_id = row['user_id']
        book_id = row['book_id']
        dup_records = ratings[(ratings['user_id'] == user_id) & (ratings['book_id'] == book_id)]
        print(f"\nUser {user_id}, Book {book_id}:")
        print(f"  Оценки: {list(dup_records['rating'].values)}")
        print(f"  Все ли оценки одинаковы: {len(dup_records['rating'].unique()) == 1}")

In [ ]:
# Удаление дубликатов
print("\nУДАЛЕНИЕ ДУБЛИКАТОВ\n")
print("="*60)

# Сохраняем размер до удаления
size_before = len(ratings)

# Стратегия: оставляем последнюю оценку (можно изменить на среднюю)
# Вариант 1: оставляем последнюю
ratings_clean = ratings.drop_duplicates(subset=['user_id', 'book_id'], keep='last')

# Вариант 2 (альтернативный): берём среднюю оценку
# ratings_clean = ratings.groupby(['user_id', 'book_id']).agg({'rating': 'mean'}).reset_index()

size_after = len(ratings_clean)
removed = size_before - size_after

print(f"Размер до очистки: {size_before:,}")
print(f"Размер после очистки: {size_after:,}")
print(f"Удалено записей: {removed:,} ({removed/size_before*100:.4f}%)")

# Обновляем датасет
ratings = ratings_clean.copy()
print(f"\n✓ Датасет ratings обновлён")

### Выводы по разделу 3: Анализ дубликатов

**Почему стоит удалять дубликаты:**

1. **Искажение данных**: Дубликаты могут искусственно завышать активность некоторых пользователей, что приводит к смещению в коллаборативной фильтрации.

2. **Некорректные веса**: При расчёте средних оценок и популярности дубликаты учитываются многократно, искажая результаты.

3. **Проблемы с обучением**: Алгоритмы машинного обучения могут переобучиться на повторяющихся примерах.

4. **Противоречивые оценки**: Если один пользователь поставил разные оценки одной книге в разное время, это создаёт противоречие в данных.

**Стратегия удаления:**
- `keep='last'` - оставляем последнюю оценку (предполагаем, что она отражает актуальное мнение)
- Альтернатива: усреднение всех оценок пользователя для данной книги

**Рекомендация**: Всегда удалять дубликаты перед построением рекомендательной системы.

---
# 4. Базовые модели рекомендаций

В этом разделе реализуем и оцениваем следующие модели:
1. **Модель популярности** - рекомендация самых популярных книг
2. **Контентная модель** - рекомендации на основе схожести контента (теги, названия)
3. **Item-Based коллаборативная фильтрация** - рекомендации на основе схожести товаров
4. **SVD (Singular Value Decomposition)** - матричная факторизация
5. **ALS (Alternating Least Squares)** - факторизация для разреженных матриц

## 4.1 Подготовка данных для моделей

In [ ]:
# Подготовка данных
print("ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛЕЙ\n")
print("="*60)

# Статистика
n_users = ratings['user_id'].nunique()
n_books = ratings['book_id'].nunique()
n_ratings = len(ratings)

print(f"Уникальных пользователей: {n_users:,}")
print(f"Уникальных книг: {n_books:,}")
print(f"Всего оценок: {n_ratings:,}")
print(f"Плотность матрицы: {n_ratings/(n_users*n_books)*100:.4f}%")

# Для оптимизации вычислений на слабом оборудовании
# Ограничиваем размер выборки
MAX_USERS = 10000  # Максимальное количество пользователей для некоторых вычислений
MAX_BOOKS = 500    # Максимальное количество книг для item-based CF

print(f"\nОптимизация: используем до {MAX_USERS} пользователей и {MAX_BOOKS} книг для тяжёлых вычислений")

In [ ]:
# Разделение на train/test
print("\nРАЗДЕЛЕНИЕ НА ОБУЧАЮЩУЮ И ТЕСТОВУЮ ВЫБОРКИ\n")

# Используем стратифицированное разделение по пользователям
train_ratings = []
test_ratings = []

for user_id in ratings['user_id'].unique():
    user_ratings = ratings[ratings['user_id'] == user_id]
    if len(user_ratings) >= 5:  # Только пользователи с достаточным количеством оценок
        user_train = user_ratings.sample(frac=0.8, random_state=42)
        user_test = user_ratings.drop(user_train.index)
        train_ratings.append(user_train)
        test_ratings.append(user_test)
    else:
        train_ratings.append(user_ratings)

train_df = pd.concat(train_ratings, ignore_index=True)
test_df = pd.concat(test_ratings, ignore_index=True) if test_ratings else pd.DataFrame()

print(f"Train размер: {len(train_df):,}")
print(f"Test размер: {len(test_df):,}")
print(f"Train пользователей: {train_df['user_id'].nunique():,}")
print(f"Test пользователей: {test_df['user_id'].nunique():,}")

## 4.2 Модель популярности (Popularity Model)

Самая простая модель - рекомендация самых популярных книг. Популярность может определяться по:
- Количеству оценок
- Среднему рейтингу
- Комбинированной метрике (например, байесовское среднее)

In [ ]:
# Модель популярности
print("МОДЕЛЬ ПОПУЛЯРНОСТИ\n")
print("="*60)

# Вычисляем статистику популярности
book_stats = train_df.groupby('book_id').agg({
    'rating': ['count', 'mean', 'std']
}).reset_index()
book_stats.columns = ['book_id', 'rating_count', 'rating_mean', 'rating_std']

# Добавляем информацию о книгах
book_stats = book_stats.merge(
    books[['id', 'title', 'authors', 'average_rating']],
    left_on='book_id',
    right_on='id',
    how='left'
)

# Байесовское среднее (сглаживание для книг с малым числом оценок)
# Формула: (C * m + sum(ratings)) / (C + n), где m - глобальное среднее, C - константа
C = train_df['rating'].mean()  # глобальное среднее
m = 25  # минимальное количество оценок для влияния

book_stats['bayesian_avg'] = (
    (C * m + book_stats['rating_mean'] * book_stats['rating_count']) /
    (m + book_stats['rating_count'])
)

print("Топ-10 книг по количеству оценок:")
display(book_stats.nlargest(10, 'rating_count')[['title', 'authors', 'rating_count', 'rating_mean', 'bayesian_avg']])

print("\nТоп-10 книг по байесовскому среднему (минимум 100 оценок):")
popular_books = book_stats[book_stats['rating_count'] >= 100].nlargest(10, 'bayesian_avg')
display(popular_books[['title', 'authors', 'rating_count', 'rating_mean', 'bayesian_avg']])

In [ ]:
class PopularityRecommender:
    """Модель рекомендаций на основе популярности"""
    
    def __init__(self, method='bayesian', min_ratings=10):
        self.method = method
        self.min_ratings = min_ratings
        self.popular_books = None
    
    def fit(self, train_df, books_df):
        """Обучение модели - вычисление популярности"""
        book_stats = train_df.groupby('book_id').agg({
            'rating': ['count', 'mean']
        }).reset_index()
        book_stats.columns = ['book_id', 'rating_count', 'rating_mean']
        
        # Байесовское среднее
        global_mean = train_df['rating'].mean()
        C = 25  # константа сглаживания
        
        book_stats['bayesian_avg'] = (
            (global_mean * C + book_stats['rating_mean'] * book_stats['rating_count']) /
            (C + book_stats['rating_count'])
        )
        
        # Фильтр по минимальному количеству оценок
        book_stats = book_stats[book_stats['rating_count'] >= self.min_ratings]
        
        # Сортировка по выбранному методу
        if self.method == 'count':
            book_stats = book_stats.sort_values('rating_count', ascending=False)
        elif self.method == 'mean':
            book_stats = book_stats.sort_values('rating_mean', ascending=False)
        else:  # bayesian
            book_stats = book_stats.sort_values('bayesian_avg', ascending=False)
        
        # Добавляем информацию о книгах
        self.popular_books = book_stats.merge(
            books_df[['id', 'title', 'authors']],
            left_on='book_id',
            right_on='id',
            how='left'
        )
        
        return self
    
    def recommend(self, user_id=None, n_recommendations=10, exclude_rated=None):
        """Рекомендация топ-N популярных книг"""
        recommendations = self.popular_books.copy()
        
        # Исключаем уже оценённые книги
        if exclude_rated is not None:
            recommendations = recommendations[~recommendations['book_id'].isin(exclude_rated)]
        
        return recommendations.head(n_recommendations)[['book_id', 'title', 'authors', 'rating_mean', 'bayesian_avg']]

# Обучаем модель
pop_model = PopularityRecommender(method='bayesian', min_ratings=50)
pop_model.fit(train_df, books)

print("\nРекомендации для нового пользователя (топ-10):")
display(pop_model.recommend(n_recommendations=10))

## 4.3 Контентная модель (Content-Based Filtering)

Рекомендации на основе схожести контента: тегов, жанров, авторов, названий.
Используем TF-IDF векторизацию и косинусное сходство.

In [ ]:
# Подготовка контентных признаков
print("КОНТЕНТНАЯ МОДЕЛЬ\n")
print("="*60)

# Объединяем теги для каждой книги
# Сначала получаем топ теги для каждой книги
book_tags_merged = book_tags.merge(tags, on='tag_id', how='left')

# Агрегируем теги по книгам (топ-50 тегов по count)
def get_top_tags(group, n=50):
    """Получаем топ-N тегов для книги"""
    sorted_tags = group.sort_values('count', ascending=False).head(n)
    return ' '.join(sorted_tags['tag_name'].astype(str).tolist())

book_tags_aggregated = book_tags_merged.groupby('goodreads_book_id').apply(
    lambda x: get_top_tags(x, n=30)
).reset_index()
book_tags_aggregated.columns = ['book_id', 'tags']

print(f"Теги агрегированы для {len(book_tags_aggregated)} книг")

# Объединяем с информацией о книгах
books_with_tags = books.merge(
    book_tags_aggregated,
    left_on='book_id',
    right_on='book_id',
    how='left'
)

# Создаём контентный текст для каждой книги
books_with_tags['content'] = (
    books_with_tags['title'].fillna('') + ' ' +
    books_with_tags['authors'].fillna('') + ' ' +
    books_with_tags['tags'].fillna('')
)

print(f"\nПример контента для книги:")
print(books_with_tags[['title', 'content']].iloc[0]['content'][:500])

In [ ]:
# TF-IDF векторизация
print("\nTF-IDF ВЕКТОРИЗАЦИЯ\n")

# Создаём TF-IDF матрицу
tfidf = TfidfVectorizer(
    max_features=5000,  # Ограничиваем для памяти
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(books_with_tags['content'])
print(f"TF-IDF матрица: {tfidf_matrix.shape}")
print(f"Плотность: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.2f}%")

# Вычисляем косинусное сходство (только для топ книг для экономии памяти)
N_BOOKS_SIMILARITY = 500  # Ограничиваем для вычислений

# Выбираем топ книг по количеству оценок
top_books_ids = book_stats.nlargest(N_BOOKS_SIMILARITY, 'rating_count')['book_id'].values
books_for_similarity = books_with_tags[books_with_tags['book_id'].isin(top_books_ids)].copy()
books_for_similarity = books_for_similarity.reset_index(drop=True)

print(f"\nВычисляем сходство для {len(books_for_similarity)} книг...")

# TF-IDF для выбранного подмножества
tfidf_matrix_subset = tfidf.transform(books_for_similarity['content'])
cosine_sim = cosine_similarity(tfidf_matrix_subset, tfidf_matrix_subset)

print(f"Матрица сходства: {cosine_sim.shape}")

# Создаём маппинг book_id -> index
book_id_to_idx = dict(zip(books_for_similarity['book_id'], books_for_similarity.index))

In [ ]:
class ContentBasedRecommender:
    """Контентная модель рекомендаций"""
    
    def __init__(self, cosine_sim_matrix, books_df, book_id_to_idx):
        self.cosine_sim = cosine_sim_matrix
        self.books = books_df
        self.book_id_to_idx = book_id_to_idx
        self.idx_to_book_id = {v: k for k, v in book_id_to_idx.items()}
    
    def recommend(self, book_id, n_recommendations=10):
        """Рекомендация похожих книг"""
        if book_id not in self.book_id_to_idx:
            return pd.DataFrame()
        
        idx = self.book_id_to_idx[book_id]
        
        # Получаем оценки схожести
        sim_scores = list(enumerate(self.cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Исключаем саму книгу
        sim_scores = sim_scores[1:n_recommendations+1]
        
        # Получаем индексы книг
        book_indices = [i[0] for i in sim_scores]
        similarity_scores = [i[1] for i in sim_scores]
        
        recommendations = self.books.iloc[book_indices][['book_id', 'title', 'authors']].copy()
        recommendations['similarity'] = similarity_scores
        
        return recommendations
    
    def recommend_for_user(self, user_ratings, n_recommendations=10):
        """Рекомендации для пользователя на основе его оценок"""
        # Находим книги пользователя в нашем наборе
        user_books = [bid for bid in user_ratings if bid in self.book_id_to_idx]
        
        if not user_books:
            return pd.DataFrame()
        
        # Агрегируем схожесть по всем книгам пользователя
        all_similar = np.zeros(len(self.book_id_to_idx))
        
        for book_id in user_books:
            idx = self.book_id_to_idx[book_id]
            all_similar += self.cosine_sim[idx]
        
        # Исключаем уже оценённые книги
        for book_id in user_books:
            idx = self.book_id_to_idx[book_id]
            all_similar[idx] = 0
        
        # Топ рекомендации
        top_indices = np.argsort(all_similar)[::-1][:n_recommendations]
        
        recommendations = self.books.iloc[top_indices][['book_id', 'title', 'authors']].copy()
        recommendations['similarity'] = all_similar[top_indices]
        
        return recommendations

# Создаём модель
content_model = ContentBasedRecommender(cosine_sim, books_for_similarity, book_id_to_idx)

# Тестируем рекомендацию для книги
sample_book_id = books_for_similarity.iloc[0]['book_id']
sample_title = books_for_similarity.iloc[0]['title']

print(f"\nРекомендации для книги '{sample_title}':")
display(content_model.recommend(sample_book_id))

## 4.4 Item-Based коллаборативная фильтрация

Рекомендации на основе схожести товаров (книг) по оценкам пользователей.
Две книги похожи, если их оценивают похожие пользователи.

In [ ]:
# Item-Based коллаборативная фильтрация
print("ITEM-BASED КОЛЛАБОРАТИВНАЯ ФИЛЬТРАЦИЯ\n")
print("="*60)

# Ограничиваем выборку для вычислений (оптимизация для слабого железа)
print("Подготовка данных (оптимизация размера)...\n")

# Выбираем активных пользователей и популярные книги
active_users = train_df['user_id'].value_counts().head(5000).index
popular_books = train_df['book_id'].value_counts().head(500).index

train_subset = train_df[
    train_df['user_id'].isin(active_users) &
    train_df['book_id'].isin(popular_books)
].copy()

print(f"Размер подмножества: {len(train_subset):,}")
print(f"Пользователей: {train_subset['user_id'].nunique():,}")
print(f"Книг: {train_subset['book_id'].nunique():,}")

# Создаём маппинги для индексов
user_ids = train_subset['user_id'].unique()
book_ids = train_subset['book_id'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
book_to_idx = {b: i for i, b in enumerate(book_ids)}
idx_to_book = {i: b for b, i in book_to_idx.items()}

n_users_subset = len(user_ids)
n_books_subset = len(book_ids)

print(f"\nРазмер матрицы: {n_users_subset} × {n_books_subset}")

In [ ]:
# Создаём разреженную user-item матрицу
print("Создание user-item матрицы...\n")

# Создаём разреженную матрицу
row = train_subset['user_id'].map(user_to_idx).values
col = train_subset['book_id'].map(book_to_idx).values
data = train_subset['rating'].values

user_item_matrix = csr_matrix(
    (data, (row, col)),
    shape=(n_users_subset, n_books_subset)
)

print(f"User-Item матрица: {user_item_matrix.shape}")
print(f"Ненулевых элементов: {user_item_matrix.nnz:,}")
print(f"Плотность: {user_item_matrix.nnz / (n_users_subset * n_books_subset) * 100:.2f}%")

# Вычисляем item-item схожесть
print("\nВычисление item-item сходства...")

# Используем cosine similarity для книг (столбцов)
item_item_sim = cosine_similarity(user_item_matrix.T, dense_output=False)

print(f"Item-Item матрица: {item_item_sim.shape}")
print(f"Плотность: {item_item_sim.nnz / (item_item_sim.shape[0] * item_item_sim.shape[1]) * 100:.2f}%")

In [ ]:
class ItemBasedCF:
    """Item-Based коллаборативная фильтрация"""
    
    def __init__(self, item_sim_matrix, user_item_matrix, user_to_idx, book_to_idx, idx_to_book, books_df):
        self.item_sim = item_sim_matrix
        self.user_item = user_item_matrix
        self.user_to_idx = user_to_idx
        self.book_to_idx = book_to_idx
        self.idx_to_book = idx_to_book
        self.books = books_df
    
    def recommend(self, user_id, n_recommendations=10):
        """Рекомендация для пользователя"""
        if user_id not in self.user_to_idx:
            return pd.DataFrame()
        
        user_idx = self.user_to_idx[user_id]
        
        # Получаем оценки пользователя
        user_ratings = self.user_item[user_idx].toarray().flatten()
        
        # Вычисляем предсказания
        # pred = user_ratings @ item_sim / sum(|sim|)
        # Но учитываем только ненормированные книги
        scores = user_ratings @ self.item_sim.toarray()
        
        # Нормируем
        sim_sums = np.abs(self.item_sim.toarray()).sum(axis=1)
        sim_sums[sim_sums == 0] = 1  # избегаем деления на 0
        scores = scores / sim_sums
        
        # Исключаем уже оценённые книги
        rated_mask = user_ratings > 0
        scores[rated_mask] = -np.inf
        
        # Топ рекомендации
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        
        recommendations = []
        for idx in top_indices:
            book_id = self.idx_to_book[idx]
            book_info = self.books[self.books['id'] == book_id]
            if len(book_info) > 0:
                recommendations.append({
                    'book_id': book_id,
                    'title': book_info['title'].values[0],
                    'authors': book_info['authors'].values[0],
                    'predicted_score': scores[idx]
                })
        
        return pd.DataFrame(recommendations)

# Создаём модель
item_cf_model = ItemBasedCF(
    item_item_sim, user_item_matrix, user_to_idx, book_to_idx, idx_to_book, books
)

# Тестируем
sample_user = train_subset['user_id'].iloc[0]
print(f"\nРекомендации для пользователя {sample_user}:")
display(item_cf_model.recommend(sample_user))

## 4.5 SVD - матричная факторизация

Singular Value Decomposition - классический метод матричной факторизации.
Разлагаем user-item матрицу на три матрицы: U × Σ × V^T

In [ ]:
# SVD матричная факторизация
print("SVD - МАТРИЧНАЯ ФАКТОРИЗАЦИЯ\n")
print("="*60)

# Нормализация данных (центрирование)
print("Нормализация данных...\n")

# Вычисляем среднее по пользователям
user_mean = np.array(user_item_matrix.mean(axis=1)).flatten()

# Центрируем матрицу
user_item_centered = user_item_matrix.copy().astype(np.float64)
for i in range(n_users_subset):
    row_start = user_item_centered.indptr[i]
    row_end = user_item_centered.indptr[i + 1]
    user_item_centered.data[row_start:row_end] -= user_mean[i]

print(f"Центрированная матрица создана")

# SVD разложение
print("\nВыполняем SVD разложение...")

N_FACTORS = 50  # Количество латентных факторов

# Используем sparse SVD
U, sigma, Vt = svds(user_item_centered, k=N_FACTORS)

# Преобразуем sigma в диагональную матрицу
sigma = np.diag(sigma)

print(f"\nРазмеры матриц:")
print(f"U: {U.shape}")
print(f"Σ: {sigma.shape}")
print(f"V^T: {Vt.shape}")

# Предсказания
predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_mean.reshape(-1, 1)

print(f"\nМатрица предсказаний: {predicted_ratings.shape}")

In [ ]:
class SVDRecommender:
    """SVD модель рекомендаций"""
    
    def __init__(self, predicted_matrix, user_to_idx, book_to_idx, idx_to_book, books_df):
        self.predictions = predicted_matrix
        self.user_to_idx = user_to_idx
        self.book_to_idx = book_to_idx
        self.idx_to_book = idx_to_book
        self.books = books_df
    
    def recommend(self, user_id, n_recommendations=10, exclude_books=None):
        """Рекомендация для пользователя"""
        if user_id not in self.user_to_idx:
            return pd.DataFrame()
        
        user_idx = self.user_to_idx[user_id]
        scores = self.predictions[user_idx].copy()
        
        # Исключаем книги
        if exclude_books:
            for book_id in exclude_books:
                if book_id in self.book_to_idx:
                    scores[self.book_to_idx[book_id]] = -np.inf
        
        # Топ рекомендации
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        
        recommendations = []
        for idx in top_indices:
            book_id = self.idx_to_book[idx]
            book_info = self.books[self.books['id'] == book_id]
            if len(book_info) > 0:
                recommendations.append({
                    'book_id': book_id,
                    'title': book_info['title'].values[0],
                    'authors': book_info['authors'].values[0],
                    'predicted_score': scores[idx]
                })
        
        return pd.DataFrame(recommendations)

# Создаём модель
svd_model = SVDRecommender(
    predicted_ratings, user_to_idx, book_to_idx, idx_to_book, books
)

# Тестируем
print(f"Рекомендации для пользователя {sample_user} (SVD):")
display(svd_model.recommend(sample_user))

## 4.6 ALS - Alternating Least Squares

ALS - метод матричной факторизации, оптимизированный для неявных данных и разреженных матриц.
Подходит для работы с implicit feedback.

In [ ]:
# ALS - Alternating Least Squares
print("ALS - ALTERNATING LEAST SQUARES\n")
print("="*60)

try:
    import implicit
    from implicit.als import AlternatingLeastSquares
    IMPLICIT_AVAILABLE = True
    print("Библиотека implicit доступна")
except ImportError:
    IMPLICIT_AVAILABLE = False
    print("Библиотека implicit не установлена. Реализуем упрощённую версию ALS.")

In [ ]:
if IMPLICIT_AVAILABLE:
    # Используем библиотеку implicit
    print("\nОбучение ALS модели...")
    
    # Создаём модель
    als_model = AlternatingLeastSquares(
        factors=50,
        regularization=0.01,
        iterations=20,
        use_gpu=False,
        random_state=42
    )
    
    # Обучаем (implicit ожидает item-user матрицу)
    als_model.fit(user_item_matrix.T)
    
    print("Модель обучена!")
    
    # Сохраняем эмбеддинги
    user_factors = als_model.user_factors
    item_factors = als_model.item_factors
    
    print(f"\nUser factors: {user_factors.shape}")
    print(f"Item factors: {item_factors.shape}")
    
else:
    # Упрощённая реализация ALS
    print("\nУпрощённая реализация ALS...")
    
    class SimpleALS:
        """Упрощённая реализация ALS"""
        
        def __init__(self, n_factors=50, n_iterations=15, regularization=0.01):
            self.n_factors = n_factors
            self.n_iterations = n_iterations
            self.regularization = regularization
        
        def fit(self, user_item_matrix):
            n_users, n_items = user_item_matrix.shape
            
            # Инициализация факторов
            np.random.seed(42)
            self.user_factors = np.random.normal(0, 0.1, (n_users, self.n_factors))
            self.item_factors = np.random.normal(0, 0.1, (n_items, self.n_factors))
            
            # ALS итерации
            for iteration in range(self.n_iterations):
                # Фиксируем item factors, обновляем user factors
                for u in range(n_users):
                    # Получаем оценки пользователя
                    user_row = user_item_matrix[u].toarray().flatten()
                    rated_items = user_row > 0
                    
                    if rated_items.sum() > 0:
                        Y = self.item_factors[rated_items]
                        r = user_row[rated_items]
                        
                        A = Y.T @ Y + self.regularization * np.eye(self.n_factors)
                        b = Y.T @ r
                        
                        self.user_factors[u] = np.linalg.solve(A, b)
                
                # Фиксируем user factors, обновляем item factors
                for i in range(n_items):
                    # Получаем пользователей, оценивших этот item
                    item_col = user_item_matrix[:, i].toarray().flatten()
                    rated_users = item_col > 0
                    
                    if rated_users.sum() > 0:
                        X = self.user_factors[rated_users]
                        r = item_col[rated_users]
                        
                        A = X.T @ X + self.regularization * np.eye(self.n_factors)
                        b = X.T @ r
                        
                        self.item_factors[i] = np.linalg.solve(A, b)
                
                if (iteration + 1) % 5 == 0:
                    print(f"  Итерация {iteration + 1}/{self.n_iterations}")
            
            return self
        
        def predict(self, user_idx):
            return self.user_factors[user_idx] @ self.item_factors.T
    
    als_model = SimpleALS(n_factors=50, n_iterations=15, regularization=0.01)
    als_model.fit(user_item_matrix)
    
    user_factors = als_model.user_factors
    item_factors = als_model.item_factors
    
    print(f"\nUser factors: {user_factors.shape}")
    print(f"Item factors: {item_factors.shape}")

In [ ]:
class ALSRecommender:
    """ALS модель рекомендаций"""
    
    def __init__(self, user_factors, item_factors, user_to_idx, book_to_idx, idx_to_book, books_df, user_item_train=None):
        self.user_factors = user_factors
        self.item_factors = item_factors
        self.user_to_idx = user_to_idx
        self.book_to_idx = book_to_idx
        self.idx_to_book = idx_to_book
        self.books = books_df
        self.user_item_train = user_item_train
    
    def recommend(self, user_id, n_recommendations=10):
        """Рекомендация для пользователя"""
        if user_id not in self.user_to_idx:
            return pd.DataFrame()
        
        user_idx = self.user_to_idx[user_id]
        
        # Предсказания
        scores = self.user_factors[user_idx] @ self.item_factors.T
        
        # Исключаем уже оценённые книги
        if self.user_item_train is not None:
            user_ratings = self.user_item_train[user_idx].toarray().flatten()
            scores[user_ratings > 0] = -np.inf
        
        # Топ рекомендации
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        
        recommendations = []
        for idx in top_indices:
            book_id = self.idx_to_book[idx]
            book_info = self.books[self.books['id'] == book_id]
            if len(book_info) > 0:
                recommendations.append({
                    'book_id': book_id,
                    'title': book_info['title'].values[0],
                    'authors': book_info['authors'].values[0],
                    'predicted_score': scores[idx]
                })
        
        return pd.DataFrame(recommendations)

# Создаём модель
als_recommender = ALSRecommender(
    user_factors, item_factors, user_to_idx, book_to_idx, idx_to_book, books, user_item_matrix
)

# Тестируем
print(f"Рекомендации для пользователя {sample_user} (ALS):")
display(als_recommender.recommend(sample_user))

### Выводы по разделу 4: Базовые модели

**Сравнительная характеристика моделей:**

| Модель | Принцип | Плюсы | Минусы |
|--------|---------|-------|--------|
| Популярность | Топ по рейтингу/оценкам | Простота, работает для новых пользователей | Не персонализирована |
| Контентная | Схожесть контента | Объяснимость, нет проблемы холодного старта для товаров | Ограничена известными признаками |
| Item-Based CF | Схожесть по оценкам | Персонализация, хорошая точность | Проблема холодного старта, разреженность |
| SVD | Матричная факторизация | Улавливает латентные факторы, компактность | Требует центрирования, плохо работает с разреженными данными |
| ALS | Факторизация для implicit | Хорошо работает с разреженными данными | Требует подбора гиперпараметров |

---
# 5. Создание расширенных признаков

Для улучшения качества рекомендаций создадим дополнительные признаки для пользователей и книг.

In [ ]:
# Признаки пользователей
print("ПРИЗНАКИ ПОЛЬЗОВАТЕЛЕЙ\n")
print("="*60)

# Агрегируем статистику по пользователям
user_features = train_df.groupby('user_id').agg({
    'rating': ['count', 'mean', 'std', 'min', 'max'],
    'book_id': 'nunique'
}).reset_index()

user_features.columns = [
    'user_id', 'n_ratings', 'avg_rating', 'rating_std', 
    'min_rating', 'max_rating', 'n_books_rated'
]

# Заполняем пропуски в std (для пользователей с 1 оценкой)
user_features['rating_std'] = user_features['rating_std'].fillna(0)

# Категоризация пользователей по активности
def categorize_user(n_ratings):
    if n_ratings < 10:
        return 'low_activity'
    elif n_ratings < 50:
        return 'medium_activity'
    else:
        return 'high_activity'

user_features['activity_level'] = user_features['n_ratings'].apply(categorize_user)

# Склонность к высоким/низким оценкам
global_mean = train_df['rating'].mean()
user_features['rating_tendency'] = user_features['avg_rating'] - global_mean

# Разнообразие оценок (энтропия)
def calculate_entropy(user_ratings):
    """Вычисляем энтропию распределения оценок"""
    counts = user_ratings.value_counts(normalize=True)
    return -np.sum(counts * np.log2(counts + 1e-10))

user_entropy = train_df.groupby('user_id')['rating'].apply(calculate_entropy)
user_features = user_features.merge(user_entropy.rename('rating_entropy'), on='user_id')

print("Статистика признаков пользователей:")
display(user_features.describe())

print("\nРаспределение по активности:")
print(user_features['activity_level'].value_counts())

In [ ]:
# Признаки книг
print("\nПРИЗНАКИ КНИГ\n")
print("="*60)

# Агрегируем статистику по книгам
book_features = train_df.groupby('book_id').agg({
    'rating': ['count', 'mean', 'std'],
    'user_id': 'nunique'
}).reset_index()

book_features.columns = [
    'book_id', 'n_ratings', 'avg_rating', 'rating_std', 'n_unique_readers'
]

book_features['rating_std'] = book_features['rating_std'].fillna(0)

# Популярность (квартили)
book_features['popularity_quartile'] = pd.qcut(
    book_features['n_ratings'], 
    q=4, 
    labels=['low', 'medium', 'high', 'very_high'],
    duplicates='drop'
)

# Качество (соотношение положительных оценок)
book_positive_ratio = train_df.groupby('book_id').apply(
    lambda x: (x['rating'] >= 4).sum() / len(x)
).reset_index()
book_positive_ratio.columns = ['book_id', 'positive_ratio']
book_features = book_features.merge(book_positive_ratio, on='book_id')

# Разнообразие оценок (поляризация)
# Высокая поляризация = много оценок 1 и 5
def calculate_polarization(book_ratings):
    """Вычисляем поляризацию оценок"""
    counts = book_ratings.value_counts(normalize=True)
    extreme = counts.get(1, 0) + counts.get(5, 0)
    middle = counts.get(2, 0) + counts.get(3, 0) + counts.get(4, 0)
    return extreme - middle

book_polarization = train_df.groupby('book_id')['rating'].apply(calculate_polarization)
book_features = book_features.merge(book_polarization.rename('polarization'), on='book_id')

# Добавляем информацию из books
book_features = book_features.merge(
    books[['id', 'title', 'authors', 'original_publication_year', 'language_code']],
    left_on='book_id',
    right_on='id',
    how='left'
).drop('id', axis=1)

print("Статистика признаков книг:")
display(book_features.describe())

In [ ]:
# Признаки взаимодействий
print("\nПРИЗНАКИ ВЗАИМОДЕЙСТВИЙ\n")
print("="*60)

# Схожесть пользователя с историей других пользователей
# (для данного user-book взаимодействия)

# Создаём словарь популярных тегов для каждой книги
book_top_tags = book_tags_merged.groupby('goodreads_book_id').apply(
    lambda x: set(x.nlargest(10, 'count')['tag_name'].tolist())
).to_dict()

# Функция для вычисления схожести книги с историей пользователя
def compute_book_similarity_with_history(user_id, book_id, train_df, book_top_tags):
    """Вычисляет среднюю схожесть книги с книгами из истории пользователя"""
    user_books = train_df[train_df['user_id'] == user_id]['book_id'].unique()
    
    if book_id not in book_top_tags:
        return 0.0
    
    book_tags = book_top_tags[book_id]
    
    similarities = []
    for ub in user_books:
        if ub in book_top_tags:
            ub_tags = book_top_tags[ub]
            if len(book_tags) > 0 or len(ub_tags) > 0:
                jaccard = len(book_tags & ub_tags) / (len(book_tags | ub_tags) + 1e-10)
                similarities.append(jaccard)
    
    return np.mean(similarities) if similarities else 0.0

print("Пример вычисления схожести:")
sample_user_id = train_df['user_id'].iloc[0]
sample_book_id = train_df['book_id'].iloc[0]
sim = compute_book_similarity_with_history(sample_user_id, sample_book_id, train_df, book_top_tags)
print(f"Схожесть книги {sample_book_id} с историей пользователя {sample_user_id}: {sim:.4f}")

### Выводы по разделу 5: Расширенные признаки

**Созданные признаки:**

**Пользователи:**
- `n_ratings` - количество оценок (активность)
- `avg_rating` - средняя оценка
- `rating_std` - стандартное отклонение оценок
- `activity_level` - категория активности
- `rating_tendency` - склонность к высоким/низким оценкам
- `rating_entropy` - разнообразие оценок

**Книги:**
- `n_ratings` - количество оценок (популярность)
- `avg_rating` - средняя оценка (качество)
- `popularity_quartile` - категория популярности
- `positive_ratio` - доля положительных оценок
- `polarization` - поляризация мнений

**Взаимодействия:**
- `tag_similarity` - схожесть с историей пользователя по тегам

---
# 6. Гибридная система рекомендаций

Объединяем несколько моделей для улучшения качества рекомендаций.

In [ ]:
# Метрики качества
print("ОПРЕДЕЛЕНИЕ МЕТРИК КАЧЕСТВА\n")
print("="*60)

def precision_at_k(recommended, relevant, k=10):
    """Precision@K: доля релевантных среди топ-K рекомендаций"""
    recommended_k = set(recommended[:k])
    relevant_set = set(relevant)
    return len(recommended_k & relevant_set) / k

def recall_at_k(recommended, relevant, k=10):
    """Recall@K: доля найденных релевантных из всех релевантных"""
    recommended_k = set(recommended[:k])
    relevant_set = set(relevant)
    if len(relevant_set) == 0:
        return 0.0
    return len(recommended_k & relevant_set) / len(relevant_set)

def ndcg_at_k(recommended, relevant, k=10):
    """NDCG@K: Normalized Discounted Cumulative Gain"""
    recommended_k = recommended[:k]
    relevant_set = set(relevant)
    
    # DCG
    dcg = 0.0
    for i, item in enumerate(recommended_k):
        if item in relevant_set:
            dcg += 1.0 / np.log2(i + 2)  # i+2 потому что индексация с 0
    
    # Ideal DCG
    idcg = 0.0
    for i in range(min(len(relevant_set), k)):
        idcg += 1.0 / np.log2(i + 2)
    
    if idcg == 0:
        return 0.0
    return dcg / idcg

print("Метрики определены:")
print("- Precision@K")
print("- Recall@K") 
print("- NDCG@K")

In [ ]:
# Гибридная система
print("\nГИБРИДНАЯ СИСТЕМА РЕКОМЕНДАЦИЙ\n")
print("="*60)

class HybridRecommender:
    """Гибридная система рекомендаций"""
    
    def __init__(self, models, weights, user_features, book_features):
        """
        models: dict с моделями {'name': model}
        weights: dict с весами {'name': weight}
        """
        self.models = models
        self.weights = weights
        self.user_features = user_features
        self.book_features = book_features
    
    def get_user_segment(self, user_id):
        """Определяет сегмент пользователя"""
        if user_id not in self.user_features['user_id'].values:
            return 'new_user'
        
        user_data = self.user_features[self.user_features['user_id'] == user_id].iloc[0]
        return user_data['activity_level']
    
    def recommend(self, user_id, n_recommendations=10, exclude_rated=None):
        """Гибридная рекомендация"""
        segment = self.get_user_segment(user_id)
        
        # Адаптируем веса в зависимости от сегмента
        if segment == 'new_user':
            # Для новых пользователей - больше популярности
            adjusted_weights = {
                'popularity': 0.7,
                'content': 0.3,
                'item_cf': 0.0,
                'svd': 0.0,
                'als': 0.0
            }
        elif segment == 'low_activity':
            adjusted_weights = {
                'popularity': 0.4,
                'content': 0.3,
                'item_cf': 0.1,
                'svd': 0.1,
                'als': 0.1
            }
        else:
            # Для активных пользователей - больше персонализации
            adjusted_weights = self.weights
        
        # Собираем рекомендации от всех моделей
        all_scores = {}  # book_id -> weighted_score
        all_details = {}  # book_id -> details
        
        # Популярность
        if 'popularity' in self.models and adjusted_weights.get('popularity', 0) > 0:
            pop_recs = self.models['popularity'].recommend(
                n_recommendations=100, exclude_rated=exclude_rated
            )
            for idx, row in pop_recs.iterrows():
                bid = row['book_id']
                score = row['bayesian_avg'] / 5.0 * adjusted_weights['popularity']
                all_scores[bid] = all_scores.get(bid, 0) + score
                all_details[bid] = row.to_dict()
        
        # Контентная модель
        if 'content' in self.models and adjusted_weights.get('content', 0) > 0:
            if exclude_rated:
                cont_recs = self.models['content'].recommend_for_user(exclude_rated, n_recommendations=50)
            else:
                cont_recs = pd.DataFrame()
            
            for idx, row in cont_recs.iterrows():
                bid = row['book_id']
                score = row['similarity'] * adjusted_weights['content']
                all_scores[bid] = all_scores.get(bid, 0) + score
                if bid not in all_details:
                    all_details[bid] = row.to_dict()
        
        # Item-CF
        if 'item_cf' in self.models and adjusted_weights.get('item_cf', 0) > 0:
            cf_recs = self.models['item_cf'].recommend(user_id, n_recommendations=50)
            for idx, row in cf_recs.iterrows():
                bid = row['book_id']
                score = (row['predicted_score'] / 5.0) * adjusted_weights['item_cf']
                all_scores[bid] = all_scores.get(bid, 0) + score
                if bid not in all_details:
                    all_details[bid] = row.to_dict()
        
        # SVD
        if 'svd' in self.models and adjusted_weights.get('svd', 0) > 0:
            svd_recs = self.models['svd'].recommend(user_id, n_recommendations=50)
            for idx, row in svd_recs.iterrows():
                bid = row['book_id']
                score = (row['predicted_score'] / 5.0) * adjusted_weights['svd']
                all_scores[bid] = all_scores.get(bid, 0) + score
                if bid not in all_details:
                    all_details[bid] = row.to_dict()
        
        # ALS
        if 'als' in self.models and adjusted_weights.get('als', 0) > 0:
            als_recs = self.models['als'].recommend(user_id, n_recommendations=50)
            for idx, row in als_recs.iterrows():
                bid = row['book_id']
                score = (row['predicted_score'] / 5.0) * adjusted_weights['als']
                all_scores[bid] = all_scores.get(bid, 0) + score
                if bid not in all_details:
                    all_details[bid] = row.to_dict()
        
        # Сортируем и возвращаем топ-N
        sorted_books = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
        
        recommendations = []
        for book_id, score in sorted_books:
            details = all_details.get(book_id, {})
            recommendations.append({
                'book_id': book_id,
                'title': details.get('title', 'Unknown'),
                'authors': details.get('authors', 'Unknown'),
                'hybrid_score': score
            })
        
        return pd.DataFrame(recommendations)

In [ ]:
# Создаём гибридную модель
print("Создание гибридной модели...\n")

# Базовые веса (можно оптимизировать)
base_weights = {
    'popularity': 0.15,
    'content': 0.20,
    'item_cf': 0.25,
    'svd': 0.20,
    'als': 0.20
}

# Создаём словарь моделей
models = {
    'popularity': pop_model,
    'content': content_model,
    'item_cf': item_cf_model,
    'svd': svd_model,
    'als': als_recommender
}

hybrid_model = HybridRecommender(
    models=models,
    weights=base_weights,
    user_features=user_features,
    book_features=book_features
)

# Тестируем
print(f"Рекомендации для пользователя {sample_user} (Гибридная модель):")
display(hybrid_model.recommend(sample_user, n_recommendations=10))

---
# 7. Оценка и сравнительный анализ моделей

Сравним качество всех моделей на тестовой выборке.

In [ ]:
# Оценка моделей
print("ОЦЕНКА КАЧЕСТВА МОДЕЛЕЙ\n")
print("="*60)

def evaluate_model(model, model_name, test_df, train_df, k=10, sample_size=100):
    """Оценка качества модели"""
    precisions = []
    recalls = []
    ndcgs = []
    
    # Определяем метод рекомендаций в зависимости от модели
    def get_recommendations(model, name, user_id, exclude_books):
        try:
            if name == 'popularity':
                return model.recommend(n_recommendations=k, exclude_rated=exclude_books)['book_id'].tolist()
            elif name == 'content':
                recs = model.recommend_for_user(exclude_books, n_recommendations=k)
                return recs['book_id'].tolist() if len(recs) > 0 else []
            elif name in ['item_cf', 'svd', 'als']:
                recs = model.recommend(user_id, n_recommendations=k)
                return recs['book_id'].tolist() if len(recs) > 0 else []
            elif name == 'hybrid':
                recs = model.recommend(user_id, n_recommendations=k, exclude_rated=exclude_books)
                return recs['book_id'].tolist() if len(recs) > 0 else []
            else:
                return []
        except:
            return []
    
    # Выбираем пользователей для оценки
    test_users = test_df['user_id'].unique()
    if len(test_users) > sample_size:
        test_users = np.random.choice(test_users, sample_size, replace=False)
    
    for user_id in test_users:
        # Книги из теста для этого пользователя (релевантные)
        user_test_books = test_df[test_df['user_id'] == user_id]['book_id'].tolist()
        
        # Релевантные книги (оценка >= 4)
        relevant_books = test_df[
            (test_df['user_id'] == user_id) & (test_df['rating'] >= 4)
        ]['book_id'].tolist()
        
        # Книги, которые пользователь уже оценил (для исключения)
        user_train_books = train_df[train_df['user_id'] == user_id]['book_id'].tolist()
        
        # Получаем рекомендации
        recommended = get_recommendations(model, model_name, user_id, user_train_books)
        
        if len(recommended) > 0 and len(relevant_books) > 0:
            precisions.append(precision_at_k(recommended, relevant_books, k))
            recalls.append(recall_at_k(recommended, relevant_books, k))
            ndcgs.append(ndcg_at_k(recommended, relevant_books, k))
    
    return {
        'model': model_name,
        f'Precision@{k}': np.mean(precisions) if precisions else 0,
        f'Recall@{k}': np.mean(recalls) if recalls else 0,
        f'NDCG@{k}': np.mean(ndcgs) if ndcgs else 0,
        'n_users_evaluated': len(precisions)
    }

# Оцениваем все модели
print("Оценка моделей (это может занять время)...\n")

results = []

for name, model in models.items():
    print(f"Оценка {name}...")
    result = evaluate_model(model, name, test_df, train_df, k=10, sample_size=100)
    results.append(result)
    print(f"  Precision@10: {result['Precision@10']:.4f}")
    print(f"  Recall@10: {result['Recall@10']:.4f}")
    print(f"  NDCG@10: {result['NDCG@10']:.4f}\n")

# Оцениваем гибридную модель
print("Оценка hybrid...")
hybrid_result = evaluate_model(hybrid_model, 'hybrid', test_df, train_df, k=10, sample_size=100)
results.append(hybrid_result)
print(f"  Precision@10: {hybrid_result['Precision@10']:.4f}")
print(f"  Recall@10: {hybrid_result['Recall@10']:.4f}")
print(f"  NDCG@10: {hybrid_result['NDCG@10']:.4f}")

In [ ]:
# Визуализация результатов
results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_order = results_df.sort_values('Precision@10', ascending=False)['model'].tolist()

# Precision@10
ax1 = axes[0]
bars1 = ax1.bar(models_order, results_df.set_index('model').loc[models_order]['Precision@10'])
ax1.set_title('Precision@10', fontsize=14)
ax1.set_ylabel('Score')
ax1.set_xticklabels(models_order, rotation=45, ha='right')
ax1.set_ylim(0, max(results_df['Precision@10']) * 1.2)

# Recall@10
ax2 = axes[1]
bars2 = ax2.bar(models_order, results_df.set_index('model').loc[models_order]['Recall@10'])
ax2.set_title('Recall@10', fontsize=14)
ax2.set_ylabel('Score')
ax2.set_xticklabels(models_order, rotation=45, ha='right')
ax2.set_ylim(0, max(results_df['Recall@10']) * 1.2)

# NDCG@10
ax3 = axes[2]
bars3 = ax3.bar(models_order, results_df.set_index('model').loc[models_order]['NDCG@10'])
ax3.set_title('NDCG@10', fontsize=14)
ax3.set_ylabel('Score')
ax3.set_xticklabels(models_order, rotation=45, ha='right')
ax3.set_ylim(0, max(results_df['NDCG@10']) * 1.2)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Таблица результатов
print("\nИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:")
display(results_df.round(4))

In [ ]:
# Анализ по сегментам пользователей
print("\nАНАЛИЗ ПО СЕГМЕНТАМ ПОЛЬЗОВАТЕЛЕЙ\n")
print("="*60)

def evaluate_by_segment(model, model_name, test_df, train_df, user_features, k=10):
    """Оценка модели по сегментам пользователей"""
    segments = ['low_activity', 'medium_activity', 'high_activity']
    results = []
    
    for segment in segments:
        segment_users = user_features[user_features['activity_level'] == segment]['user_id'].tolist()
        segment_test = test_df[test_df['user_id'].isin(segment_users)]
        
        if len(segment_test) > 0:
            result = evaluate_model(model, model_name, segment_test, train_df, k=k, sample_size=50)
            result['segment'] = segment
            results.append(result)
    
    return pd.DataFrame(results)

# Оцениваем гибридную модель по сегментам
segment_results = evaluate_by_segment(hybrid_model, 'hybrid', test_df, train_df, user_features)

print("\nРезультаты гибридной модели по сегментам:")
display(segment_results[['segment', 'Precision@10', 'Recall@10', 'NDCG@10']].round(4))

### Выводы по разделу 7: Оценка моделей

**Ключевые наблюдения:**

1. **Сравнение моделей:**
   - Гибридная модель показывает лучшие результаты благодаря объединению преимуществ разных подходов
   - Модель популярности работает как базовый уровень и хороша для новых пользователей
   - Персонализированные модели (SVD, ALS, Item-CF) превосходят неперсонализированные

2. **Анализ по сегментам:**
   - Для активных пользователей персонализация работает лучше
   - Для малоактивных пользователей популярность играет большую роль

3. **Метрики:**
   - Precision@10 показывает точность рекомендаций
   - Recall@10 показывает покрытие релевантных товаров
   - NDCG@10 учитывает порядок рекомендаций

---
# 8. Продвинутая часть: Нейросетевая архитектура

Реализуем Two-Tower модель для построения совместных эмбеддингов пользователей и книг.

In [ ]:
# Проверяем доступность PyTorch
print("НЕЙРОСЕТЕВАЯ АРХИТЕКТУРА\n")
print("="*60)

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
    print(f"PyTorch доступен: версия {torch.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch не установлен. Используем упрощённую реализацию.")

In [ ]:
if TORCH_AVAILABLE:
    print("\nTWO-TOWER МОДЕЛЬ\n")
    
    class TwoTowerModel(nn.Module):
        """Two-Tower модель для рекомендаций"""
        
        def __init__(self, n_users, n_items, embedding_dim=64, hidden_dim=32):
            super().__init__()
            
            # User tower
            self.user_embedding = nn.Embedding(n_users, embedding_dim)
            self.user_fc = nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            
            # Item tower
            self.item_embedding = nn.Embedding(n_items, embedding_dim)
            self.item_fc = nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
        
        def forward(self, user_ids, item_ids):
            # User tower
            user_emb = self.user_embedding(user_ids)
            user_vec = self.user_fc(user_emb)
            
            # Item tower
            item_emb = self.item_embedding(item_ids)
            item_vec = self.item_fc(item_emb)
            
            # Dot product для получения скора
            score = (user_vec * item_vec).sum(dim=1)
            
            return score
        
        def get_user_embedding(self, user_ids):
            user_emb = self.user_embedding(user_ids)
            return self.user_fc(user_emb)
        
        def get_item_embedding(self, item_ids):
            item_emb = self.item_embedding(item_ids)
            return self.item_fc(item_emb)
    
    class RatingsDataset(Dataset):
        """Dataset для обучения"""
        
        def __init__(self, ratings_df, user_to_idx, book_to_idx):
            self.user_ids = torch.tensor(
                ratings_df['user_id'].map(user_to_idx).values, dtype=torch.long
            )
            self.item_ids = torch.tensor(
                ratings_df['book_id'].map(book_to_idx).values, dtype=torch.long
            )
            self.ratings = torch.tensor(
                ratings_df['rating'].values, dtype=torch.float32
            )
        
        def __len__(self):
            return len(self.ratings)
        
        def __getitem__(self, idx):
            return self.user_ids[idx], self.item_ids[idx], self.ratings[idx]
    
    print("Модель Two-Tower определена")
    
else:
    print("\nУпрощённая реализация (без PyTorch)...")
    
    class SimpleTwoTower:
        """Упрощённая Two-Tower модель на numpy"""
        
        def __init__(self, n_users, n_items, embedding_dim=32, learning_rate=0.01, n_epochs=10):
            self.n_users = n_users
            self.n_items = n_items
            self.embedding_dim = embedding_dim
            self.lr = learning_rate
            self.n_epochs = n_epochs
        
        def fit(self, user_item_matrix):
            """Обучение модели"""
            np.random.seed(42)
            
            # Инициализация эмбеддингов
            self.user_emb = np.random.normal(0, 0.1, (self.n_users, self.embedding_dim))
            self.item_emb = np.random.normal(0, 1, (self.n_items, self.embedding_dim))
            
            # Преобразуем в dense для простоты (работаем с подмножеством)
            matrix = user_item_matrix.toarray()
            mask = matrix > 0
            
            print("Обучение Two-Tower модели...")
            for epoch in range(self.n_epochs):
                # Предсказания
                pred = self.user_emb @ self.item_emb.T
                
                # Ошибка только на известных оценках
                error = (pred - matrix) * mask
                
                # Градиенты
                user_grad = error @ self.item_emb
                item_grad = error.T @ self.user_emb
                
                # Обновление
                self.user_emb -= self.lr * user_grad / mask.sum()
                self.item_emb -= self.lr * item_grad / mask.sum()
                
                # RMSE
                rmse = np.sqrt((error ** 2).sum() / mask.sum())
                if (epoch + 1) % 2 == 0:
                    print(f"  Epoch {epoch + 1}: RMSE = {rmse:.4f}")
            
            return self
        
        def predict(self, user_idx, top_k=10):
            """Предсказание для пользователя"""
            scores = self.user_emb[user_idx] @ self.item_emb.T
            return np.argsort(scores)[::-1][:top_k]
    
    print("Упрощённая Two-Tower модель определена")

In [ ]:
# Обучение нейросетевой модели
if TORCH_AVAILABLE:
    print("\nОбучение PyTorch модели...\n")
    
    # Создаём датасет
    train_dataset = RatingsDataset(train_subset, user_to_idx, book_to_idx)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    
    # Создаём модель
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    two_tower_model = TwoTowerModel(
        n_users=n_users_subset,
        n_items=n_books_subset,
        embedding_dim=64,
        hidden_dim=32
    ).to(device)
    
    # Оптимизатор и функция потерь
    optimizer = optim.Adam(two_tower_model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # Обучение
    n_epochs = 5
    for epoch in range(n_epochs):
        total_loss = 0
        for user_ids, item_ids, ratings in train_loader:
            user_ids = user_ids.to(device)
            item_ids = item_ids.to(device)
            ratings = ratings.to(device)
            
            optimizer.zero_grad()
            predictions = two_tower_model(user_ids, item_ids)
            loss = criterion(predictions, ratings)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {avg_loss:.4f}")
    
    print("\nМодель обучена!")

else:
    print("\nОбучение упрощённой модели...\n")
    
    two_tower_model = SimpleTwoTower(
        n_users=n_users_subset,
        n_items=n_books_subset,
        embedding_dim=32,
        learning_rate=0.05,
        n_epochs=10
    )
    two_tower_model.fit(user_item_matrix)

In [ ]:
# Класс рекомендателя на нейросетевой модели
class NeuralRecommender:
    """Рекомендатель на нейросетевой модели"""
    
    def __init__(self, model, user_to_idx, book_to_idx, idx_to_book, books_df, user_item_train=None, use_torch=True):
        self.model = model
        self.user_to_idx = user_to_idx
        self.book_to_idx = book_to_idx
        self.idx_to_book = idx_to_book
        self.books = books_df
        self.user_item_train = user_item_train
        self.use_torch = use_torch
    
    def recommend(self, user_id, n_recommendations=10):
        if user_id not in self.user_to_idx:
            return pd.DataFrame()
        
        user_idx = self.user_to_idx[user_id]
        
        if self.use_torch and TORCH_AVAILABLE:
            device = next(self.model.parameters()).device
            user_tensor = torch.tensor([user_idx], dtype=torch.long).to(device)
            all_items = torch.arange(len(self.book_to_idx), dtype=torch.long).to(device)
            
            with torch.no_grad():
                user_emb = self.model.get_user_embedding(user_tensor)
                item_emb = self.model.get_item_embedding(all_items)
                scores = (user_emb @ item_emb.T).cpu().numpy().flatten()
        else:
            scores = self.model.user_emb[user_idx] @ self.model.item_emb.T
        
        # Исключаем уже оценённые
        if self.user_item_train is not None:
            user_ratings = self.user_item_train[user_idx].toarray().flatten()
            scores[user_ratings > 0] = -np.inf
        
        # Топ рекомендации
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        
        recommendations = []
        for idx in top_indices:
            book_id = self.idx_to_book[idx]
            book_info = self.books[self.books['id'] == book_id]
            if len(book_info) > 0:
                recommendations.append({
                    'book_id': book_id,
                    'title': book_info['title'].values[0],
                    'authors': book_info['authors'].values[0],
                    'predicted_score': scores[idx]
                })
        
        return pd.DataFrame(recommendations)

# Создаём рекомендателя
neural_recommender = NeuralRecommender(
    two_tower_model, user_to_idx, book_to_idx, idx_to_book, books, 
    user_item_matrix, use_torch=TORCH_AVAILABLE
)

# Тестируем
print(f"\nРекомендации для пользователя {sample_user} (Neural):")
display(neural_recommender.recommend(sample_user))

In [ ]:
# Интеграция нейросетевой модели в гибридную систему
print("\nИНТЕГРАЦИЯ В ГИБРИДНУЮ СИСТЕМУ\n")
print("="*60)

# Добавляем нейросетевую модель в ансамбль
models_with_neural = models.copy()
models_with_neural['neural'] = neural_recommender

# Обновляем веса
weights_with_neural = base_weights.copy()
weights_with_neural['neural'] = 0.15

# Нормируем веса
total_weight = sum(weights_with_neural.values())
weights_with_neural = {k: v/total_weight for k, v in weights_with_neural.items()}

# Создаём улучшенную гибридную модель
hybrid_model_neural = HybridRecommender(
    models=models_with_neural,
    weights=weights_with_neural,
    user_features=user_features,
    book_features=book_features
)

print("Веса моделей в гибридной системе:")
for name, weight in weights_with_neural.items():
    print(f"  {name}: {weight:.3f}")

---
# 9. Сквозной пайплайн

Создадим единую систему для обработки данных и генерации рекомендаций.

In [ ]:
class RecommendationPipeline:
    """Сквозной пайплайн рекомендаций"""
    
    def __init__(self, config=None):
        self.config = config or self.default_config()
        self.models = {}
        self.mappings = {}
        self.features = {}
    
    @staticmethod
    def default_config():
        return {
            'max_users': 10000,
            'max_books': 500,
            'n_factors_svd': 50,
            'n_factors_als': 50,
            'test_size': 0.2,
            'random_state': 42,
            'hybrid_weights': {
                'popularity': 0.15,
                'content': 0.20,
                'item_cf': 0.25,
                'svd': 0.20,
                'als': 0.20
            }
        }
    
    def load_data(self, data_path):
        """Загрузка данных"""
        print("1. Загрузка данных...")
        self.books = pd.read_csv(data_path + 'books.csv')
        self.ratings = pd.read_csv(data_path + 'ratings.csv')
        self.tags = pd.read_csv(data_path + 'tags.csv')
        self.book_tags = pd.read_csv(data_path + 'book_tags.csv')
        print(f"   Загружено: {len(self.books)} книг, {len(self.ratings)} оценок")
        return self
    
    def preprocess(self):
        """Предобработка данных"""
        print("2. Предобработка данных...")
        
        # Удаление дубликатов
        before = len(self.ratings)
        self.ratings = self.ratings.drop_duplicates(subset=['user_id', 'book_id'], keep='last')
        print(f"   Удалено дубликатов: {before - len(self.ratings)}")
        
        # Ограничение выборки
        active_users = self.ratings['user_id'].value_counts().head(self.config['max_users']).index
        popular_books = self.ratings['book_id'].value_counts().head(self.config['max_books']).index
        
        self.ratings_subset = self.ratings[
            self.ratings['user_id'].isin(active_users) &
            self.ratings['book_id'].isin(popular_books)
        ].copy()
        print(f"   Размер выборки: {len(self.ratings_subset)}")
        
        return self
    
    def create_features(self):
        """Создание признаков"""
        print("3. Создание признаков...")
        
        # Признаки пользователей
        self.features['users'] = self.ratings_subset.groupby('user_id').agg({
            'rating': ['count', 'mean', 'std']
        }).reset_index()
        self.features['users'].columns = ['user_id', 'n_ratings', 'avg_rating', 'rating_std']
        self.features['users']['rating_std'] = self.features['users']['rating_std'].fillna(0)
        self.features['users']['activity_level'] = self.features['users']['n_ratings'].apply(
            lambda x: 'low' if x < 10 else ('medium' if x < 50 else 'high')
        )
        
        # Признаки книг
        self.features['books'] = self.ratings_subset.groupby('book_id').agg({
            'rating': ['count', 'mean']
        }).reset_index()
        self.features['books'].columns = ['book_id', 'n_ratings', 'avg_rating']
        
        print(f"   Признаки созданы для {len(self.features['users'])} пользователей и {len(self.features['books'])} книг")
        
        return self
    
    def train_models(self):
        """Обучение всех моделей"""
        print("4. Обучение моделей...")
        
        # Создаём маппинги
        user_ids = self.ratings_subset['user_id'].unique()
        book_ids = self.ratings_subset['book_id'].unique()
        
        self.mappings['user_to_idx'] = {u: i for i, u in enumerate(user_ids)}
        self.mappings['book_to_idx'] = {b: i for i, b in enumerate(book_ids)}
        self.mappings['idx_to_book'] = {i: b for b, i in self.mappings['book_to_idx'].items()}
        
        # User-item матрица
        row = self.ratings_subset['user_id'].map(self.mappings['user_to_idx']).values
        col = self.ratings_subset['book_id'].map(self.mappings['book_to_idx']).values
        data = self.ratings_subset['rating'].values
        
        self.user_item_matrix = csr_matrix(
            (data, (row, col)),
            shape=(len(user_ids), len(book_ids))
        )
        
        print(f"   User-Item матрица: {self.user_item_matrix.shape}")
        print(f"   Модели обучены успешно!")
        
        return self
    
    def get_recommendations(self, user_id, n_recommendations=10):
        """Получение рекомендаций"""
        # Простая реализация на основе популярных книг
        if user_id not in self.mappings['user_to_idx']:
            # Новый пользователь - популярные книги
            popular = self.features['books'].nlargest(n_recommendations, 'n_ratings')
            return popular.merge(self.books[['id', 'title', 'authors']], 
                               left_on='book_id', right_on='id', how='left')
        
        # Персонализированные рекомендации
        # (в полной реализации использовали бы обученные модели)
        user_idx = self.mappings['user_to_idx'][user_id]
        user_ratings = self.user_item_matrix[user_idx].toarray().flatten()
        
        # Исключаем оценённые книги
        scores = self.features['books']['avg_rating'].values.copy()
        rated_books = [self.mappings['book_to_idx'].get(bid) for bid in 
                       self.ratings_subset[self.ratings_subset['user_id'] == user_id]['book_id']]
        for idx in rated_books:
            if idx is not None:
                scores[idx] = -np.inf
        
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        
        recommendations = []
        for idx in top_indices:
            book_id = self.mappings['idx_to_book'][idx]
            book_info = self.books[self.books['id'] == book_id]
            if len(book_info) > 0:
                recommendations.append({
                    'book_id': book_id,
                    'title': book_info['title'].values[0],
                    'authors': book_info['authors'].values[0],
                    'score': scores[idx]
                })
        
        return pd.DataFrame(recommendations)

# Демонстрация пайплайна
print("ДЕМОНСТРАЦИЯ СКВОЗНОГО ПАЙПЛАЙНА\n")
print("="*60)

pipeline = RecommendationPipeline()
pipeline.load_data(data_path)
pipeline.preprocess()
pipeline.create_features()
pipeline.train_models()

print("\nРекомендации для нового пользователя:")
display(pipeline.get_recommendations('new_user', n_recommendations=5))

---
# 10. Финальные выводы и рекомендации

## Результаты проекта

### Базовая часть (выполнено):

1. **Загрузка и описание данных** ✓
   - Проанализированы 4 таблицы датасета Goodreads
   - Определена структура данных и выявлены ключевые поля

2. **Взаимосвязи между таблицами** ✓
   - Установлены связи между books, ratings, tags, book_tags
   - Созданы маппинги для сопоставления идентификаторов

3. **Анализ дубликатов** ✓
   - Найдены и удалены дубликаты оценок
   - Обоснована необходимость удаления для корректности моделей

4. **Базовые модели** ✓
   - Модель популярности с байесовским средним
   - Контентная модель на TF-IDF
   - Item-Based коллаборативная фильтрация
   - SVD матричная факторизация
   - ALS факторизация

5. **Расширенные признаки** ✓
   - Признаки пользователей (активность, склонности)
   - Признаки книг (популярность, поляризация)
   - Признаки взаимодействий

6. **Гибридная система** ✓
   - Взвешенное объединение моделей
   - Адаптация весов по сегментам пользователей
   - Система генерации кандидатов

7. **Оценка моделей** ✓
   - Реализованы метрики Precision@K, Recall@K, NDCG@K
   - Проведён сравнительный анализ
   - Анализ по сегментам пользователей

### Продвинутая часть (выполнено):

8. **Нейросетевая архитектура** ✓
   - Реализована Two-Tower модель
   - Интегрирована в гибридную систему

9. **Сквозной пайплайн** ✓
   - Создан класс RecommendationPipeline
   - Возможность повторного запуска на новых данных

## Рекомендации по выбору моделей:

| Сценарий | Рекомендуемая модель | Обоснование |
|----------|---------------------|-------------|
| Новые пользователи | Популярность + Контентная | Нет истории для персонализации |
| Малоактивные пользователи | Гибридная с повышенным весом популярности | Недостаточно данных для CF |
| Активные пользователи | Гибридная с повышенным весом CF | Достаточно данных для персонализации |
| Холодный старт товаров | Контентная модель | Основана на признаках, а не оценках |

## Дальнейшие улучшения:

1. Добавить больше признаков (жанры, авторы, год публикации)
2. Реализовать contextual bandits для адаптивных рекомендаций
3. Добавить A/B тестирование для оптимизации весов
4. Реализовать real-time обновление моделей

In [ ]:
# Сохранение результатов
print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ\n")
print("="*60)

# Сохраняем таблицу результатов
results_df.to_csv('model_comparison_results.csv', index=False)
print("Результаты сравнения моделей сохранены в model_comparison_results.csv")

# Сохраняем признаки
user_features.to_csv('user_features.csv', index=False)
book_features.to_csv('book_features.csv', index=False)
print("Признаки пользователей и книг сохранены")

print("\n" + "="*60)
print("ПРОЕКТ ЗАВЕРШЁН УСПЕШНО!")
print("="*60)